In [1]:
!nvidia-smi

Mon Sep  8 12:53:59 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.07             Driver Version: 570.133.07     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        Off |   00000000:04:00.0 Off |                  N/A |
| 30%   28C    P8             20W /  350W |   23559MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  
import numpy as np
import torch, faiss


device = torch.device("cuda:0")
torch.cuda.set_device(device)



In [3]:
import os
import torch
import pandas as pd
import pyterrier as pt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Initialize PyTerrier
if not pt.started():
    pt.init()


/tmp/ipykernel_173058/896768055.py:9: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_173058/896768055.py:10: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [4]:
# Step 1: Load dataset
dataset = pt.get_dataset("msmarco_passage")
topics19 = dataset.get_topics("test-2019")
qrels2019 = dataset.get_qrels("test-2019")
topics2019 = topics19.merge(qrels2019, on = "qid")[["qid","query"]].drop_duplicates()
topics20 = dataset.get_topics("test-2020")
qrels2020 = dataset.get_qrels("test-2020")
topics2020 = topics20.merge(qrels2020, on = "qid")[["qid","query"]].drop_duplicates()
topics_dlhard = pt.get_dataset('irds:msmarco-passage/trec-dl-hard').get_topics()
qrels_dlhard = pt.get_dataset('irds:msmarco-passage/trec-dl-hard').get_qrels()

In [5]:
# from pyterrier_colbert.ranking import ColBERTv2Index
# import time
# start_time = time.time()
# colbert_model_path = '/data2/wangxiao/pyterrier_colbert/colbertv2.0'
# # bm25_terrier_stemmed = pt.BatchRetrieve.from_dataset('vaswani', 'terrier_stemmed', wmodel='BM25')
# factory = ColBERTv2Index(colbert=colbert_model_path, 
#                          index_location="/data2/wangxiao/pyterrier_colbert2/msmarco_psg_v1_index/2bits/indexes/2bits/",
#                            plaid_mode=True, ncells=2, centroid_score_threshold=0.45, ndocs=1024) 
# # plaid_e2e = factory.plaid_end_to_end(k=1000)
# end_time = time.time()

# elapsed_time = end_time - start_time
# print(f" V2 retrieval time: {elapsed_time:.2f} seconds")

In [6]:
from pyterrier_colbert.ranking import ColBERTv2Index
import time
start_time = time.time()
colbert_model_path = '/data2/wangxiao/pyterrier_colbert/colbertv2.0'
# bm25_terrier_stemmed = pt.BatchRetrieve.from_dataset('vaswani', 'terrier_stemmed', wmodel='BM25')
factory = ColBERTv2Index(colbert=colbert_model_path, 
                         index_location="/data2/wangxiao/pyterrier_colbert2/msmarco_psg_v1_index/2bits/indexes/2bits/",
                           plaid_mode=True, ncells=4, centroid_score_threshold=0.4, ndocs=4096) 
# plaid_e2e = factory.plaid_end_to_end(k=1000)
end_time = time.time()

elapsed_time = end_time - start_time
print(f" V2 retrieval time: {elapsed_time:.2f} seconds")

[Sep 08, 12:56:09] #> Loading codec...
[Sep 08, 12:56:09] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


/data2/wangxiao/anaconda3/envs/v2/lib/python3.8/site-packages/torch/utils/cpp_extension.py:1965: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(


[Sep 08, 12:56:10] Loading packbits_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


/data2/wangxiao/anaconda3/envs/v2/lib/python3.8/site-packages/torch/utils/cpp_extension.py:1965: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(


[Sep 08, 12:56:10] #> Loading IVF...
[Sep 08, 12:56:14] #> Loading doclens...


100%|███████████████████████████████████████| 354/354 [00:00<00:00, 1297.50it/s]


[Sep 08, 12:56:16] #> Loading codes and residuals...


100%|█████████████████████████████████████████| 354/354 [01:04<00:00,  5.45it/s]


 V2 retrieval time: 179.52 seconds


# One-time running:
- 1. obtain the global IDF score
  2. obtain the stopwords' corresponding ids

In [123]:
import os
import glob
import json
import math
from collections import Counter
from tqdm import tqdm

import torch
from colbert.infra import ColBERTConfig
from colbert.searcher import Searcher


# INDEX_ROOT = "/data2/wangxiao/pyterrier_colbert2/msmarco_psg_v1_index/",
# INDEX_NAME = "2bits/indexes/2bits"
# FULL_INDEX = os.path.join(INDEX_ROOT, INDEX_NAME)
FULL_INDEX = '/data2/wangxiao/pyterrier_colbert2/msmarco_psg_v1_index/2bits/indexes/2bits/'

# 1) calcualte num_docs
num_docs = 0
for meta_path in glob.glob(os.path.join(FULL_INDEX, "*.metadata.json")):
    with open(meta_path) as f:
        meta = json.load(f)
        num_docs += meta.get("num_passages", 0)


# 2) collect all shards
shard_prefixes = sorted({
    fname.split(".", 1)[0]
    for fname in os.listdir(FULL_INDEX)
    if fname.split(".", 1)[0].isdigit()
})
print(f"shards num all: {len(shard_prefixes)}")

# 3) load colbert settings
colbert_cfg = ColBERTConfig.load_from_checkpoint("colbert-ir/colbertv2.0")

# 3) calculate df
df_counts = Counter()

# for prefix in shard_prefixes:
for prefix in tqdm(shard_prefixes, desc="traverse all shards..."):
    # 3.1) load current shard's doc lengths
    lens_path = os.path.join(FULL_INDEX, f"doclens.{prefix}.json")
    lengths   = json.load(open(lens_path, 'r', encoding='utf8'))  # list of ints
    local_docs = len(lengths)

    # 3.2) load shard's codes sequence(i.e. all token\s embedding IDs）
    codes_path = os.path.join(FULL_INDEX, f"{prefix}.codes.pt")
    codes      = torch.load(codes_path)  # 1D LongTensor, sum(lengths) 
    # 3.3) calculate every doc in codes  offset
    offsets = [0]
    for l in lengths[:-1]:
        offsets.append(offsets[-1] + l)
    # 3.4) process doc_id
    for doc_id, offset in enumerate(offsets):
        l = lengths[doc_id]
        token_ids = codes[offset: offset + l]      # doc's all embedding IDs
        df_counts.update(set(token_ids.tolist()))  # set() make sure unique id

print(f" {len(df_counts)} embedding IDs DF")

# 4) calucate IDF
idf_map = {    eid: math.log((num_docs + 1) / (df + 1)) + 1    for eid, df in df_counts.items()
}
print("Some IDF values:")
for eid in list(idf_map)[:5]:
    print(f"  emb_id={eid:6d}  df={df_counts[eid]:7d}  idf={idf_map[eid]:.4f}")

shards num all: 354


traverse all shards...: 100%|█████████████████| 354/354 [03:03<00:00,  1.93it/s]

 262144 embedding IDs DF
Some IDF values:
  emb_id=233603  df=   3706  idf=8.7770
  emb_id= 12688  df=   1609  idf=9.6110
  emb_id=211481  df=   1854  idf=9.4694
  emb_id=211098  df=   3633  idf=8.7969
  emb_id=210301  df=   3530  idf=8.8257


In [8]:
import json

with open("./idf_map.json", "r", encoding="utf-8") as f:
    idf_map = json.load(f)

# obtain the stopword's corresponding ids, then we can filter them before expansion

In [20]:
import os, numpy as np, torch, json
from pathlib import Path
from colbert.modeling.tokenization import DocTokenizer

def build_stop_wp_ids(dtok, stop_words, drop_subwords=False):
    """
    mapping the stopwords to a wordpiece token id set. define what type of tokens in the set
    把停用词映射为 WP id 集合。若 drop_subwords=True，可额外剔除 '##' 前缀子词。
    """
    wp_ids = set()
    for w in stop_words:
        # using the Doctokenizer
        toks = dtok.tok.tokenize(w)
        ids  = dtok.tok.convert_tokens_to_ids(toks)
        for i, t in zip(ids, toks):
            if drop_subwords and t.startswith('##'): 
                continue
            wp_ids.add(int(i))
    return wp_ids

class WPSidecarWriter:
    """
    把每个 pid 的 WordPiece ids 顺序写入一个连续的 .bin 文件，
    同时写出 offsets 以便 O(1) 随机访问（与 embeddings_strided 的顺序对齐）。
    """
    def __init__(self, out_dir: str):
        self.out_dir = Path(out_dir); self.out_dir.mkdir(parents=True, exist_ok=True)
        self.ids_path = self.out_dir / "wp_ids.int32.bin"
        self.off_path = self.out_dir / "offsets.int64.npy"
        self._ids = []   # 临时缓存，最后统一写入（也可直接用mmap增量写）
        self._offs = [0] # offsets[pid] 起点，下标长度 = N_docs+1

    def append_doc(self, wp_ids_1d):
        self._ids.append(np.asarray(wp_ids_1d, dtype=np.int32))
        self._offs.append(self._offs[-1] + len(wp_ids_1d))

    def finalize(self):
        ids = np.concatenate(self._ids, axis=0) if len(self._ids)>0 else np.zeros((0,), np.int32)
        offs = np.asarray(self._offs, dtype=np.int64)
        ids.tofile(self.ids_path.open("wb"))
        np.save(self.off_path, offs)

def dump_wp_sidecar(dataset, factory, out_dir, drop_subwords=False):
    """
    Using the same Doctokenizer, 
    establish the pid-> wordpiece tokenid mapping.  
    Key point: Using L in the embeddings_strided, as the cutoff for each passage.
    
    """
    cfg  = getattr(factory.searcher, "config", None) or getattr(factory.searcher.ranker, "config", None)
    dtok = DocTokenizer(config=cfg)
    embS = factory.searcher.ranker.embeddings_strided
    docnos = factory.docnos

    # corpus Iteration
    
    df = dataset.get_corpus()
    rows = ({"docno": r["docno"], "text": r["text"]} for _, r in df.iterrows())
    corpus_iter = list(rows)  # 


    writer = WPSidecarWriter(out_dir)
    N = len(docnos)  # all the docnos, all the pids
    
    # Establish docno -> text mapping, then dump using pid 
    doc_map = {rec["docno"]: rec["text"] for rec in corpus_iter}

    for pid in range(N):
        docno = docnos.fwd[int(pid)]
        text  = doc_map.get(docno, None)
        if text is None:
            writer.append_doc([]); 
            continue

        # keep align using L
        # 与索引对齐的长度 L（最好直接用 doclens；若没有，可仅取 lengths）
        _, lengths = embS.lookup_pids([pid])  # 只取长度，避免大规模解压
        L = int(lengths[0]) if lengths is not None and len(lengths)>0 else None

        ids, _ = dtok.tensorize([text])        # 与索引相同的tokenizer/规则
        wp_ids = ids[0].tolist()
        if L is not None: 
            wp_ids = wp_ids[:L]                # 截断对齐到嵌入长度
        writer.append_doc(wp_ids)

    writer.finalize()
    return str(writer.ids_path), str(writer.off_path)

# —— Only run one-time 

def get_cfg(searcher):
    return (getattr(searcher, "config", None)
            or getattr(searcher.ranker, "config", None)
            or getattr(searcher.ranker, "colbert_config", None))


STOP = {"the","and","of","to","in","a","is","it","that","as","for","its","an","on","are","was","be","by",
        "with","or","this","from","at","which","we","have","has","had","were","but","not","can","their",
        "there","they","he","she","you","i","my","our","your","his","her","them","us","me","do","did",
        "does","been","being","so","if","then","than","also","into","out","about","over","under","up",
        "down","no","yes","all","any","some","most","more","such","per","via","vs","etc"}
cfg = get_cfg(factory.searcher)
dtok = DocTokenizer(config=cfg)
STOP_WP_IDS = build_stop_wp_ids(dtok, STOP, drop_subwords=True)
ids_path, off_path = dump_wp_sidecar(dataset, factory, out_dir="./wp_sidecar", drop_subwords=True)
with open("./wp_sidecar_subword/stop_wp_ids.json", "w") as f: json.dump(sorted(list(STOP_WP_IDS)), f)


In [12]:
!ls ./wp_sidecar/wp_ids.int32.bin

offsets.int64.npy  stop_wp_ids.json  wp_ids.int32.bin


In [10]:
class WPSidecar:
    def __init__(self, ids_path, off_path):
        self.ids_mmap = np.memmap(ids_path, mode="r", dtype=np.int32)
        self.offsets  = np.load(off_path)  # int64, 长度N+1
    def lookup_wpids(self, pids):
        # 返回拼接的一维 WP id 张量 + 对应的 doc 长度列表
        out = []
        lens = []
        for pid in pids:
            s = int(self.offsets[pid]); e = int(self.offsets[pid+1])
            arr = self.ids_mmap[s:e]
            out.append(torch.from_numpy(arr.copy()))  # 小切片copy避免引用mmap句柄
            lens.append(e - s)
        return (torch.cat(out, dim=0) if out else torch.empty(0, dtype=torch.int64)), lens

In [13]:
import json

ids_path = "./wp_sidecar/wp_ids.int32.bin"
off_path = "./wp_sidecar/offsets.int64.npy"
from pathlib import Path
STOP_WP_IDS = set(json.load(open("./wp_sidecar/stop_wp_ids.json")))
wp_sidecar   = WPSidecar(ids_path, off_path)



In [14]:
# # with open("./idf_map.json", "w", encoding = "utf-8") as f:
# #     json.dump(idf_map, f, indent = 4)
# import json
# with open("./idf_map.json", "r", encoding="utf-8") as f:
#     idf_map = json.load(f)

# below is Query Expansion Implementation:
- step1: obtain PRF docs from dense_search;
- step2: obtain the pid's embeddings;
- step3: perfrom some filter then importance scoring;
- step4: perform query expansion

In [16]:
import torch.nn.functional as F
qtext= "do goldfish grow"
Q = factory.searcher.encode([qtext]).squeeze(0)     # [Lq, dim]
Q = F.normalize(Q, p=2, dim=-1)
pids, _, _ = factory.searcher.dense_search(Q.unsqueeze(0), k=3)
pids

[2928707, 8182159, 1960255]

In [17]:
embS = factory.searcher.ranker.embeddings_strided
V, lens  = embS.lookup_pids(pids)
codes, _ = embS.lookup_codes([pids[1]])


In [18]:
codes

tensor([ 38240,  86639, 115035, 254466,  99759, 254466, 165187, 220038,  95511,
         25450,  93334,  38240, 242603, 242193, 180219, 167835,  29000, 218151,
        161871,   7971, 134320,  29000,  29000, 224825,  16866, 161464, 253601,
         85747,  67063, 244873, 157876, 213673,  10712,  98856,  93728, 234148,
         38240,  84268,  91269,  94232,  64546,  38240, 135633,  93728, 212414,
         70566, 209018,  38240, 165187, 220038, 214383, 106446, 214383,  24284,
         62050, 185231, 204145, 143727,  38240,  66934, 176909,  38240, 214112,
        128271, 150710,  62050,  90824, 198509, 120691,  82094,  38240, 204145,
         38240], device='cuda:0', dtype=torch.int32)

In [19]:
wpids, _ = wp_sidecar.lookup_wpids([pids[1]])
wpids

tensor([  101,     2,  1005,  2751,  7529,  2788,  4982,  2007,  2037,  4044,
         2061,  2065,  2027,  2024,  1999,  1037,  2312,  2686,  1998,  2024,
         7349,  2438,  1010,  2027,  2079,  2823,  2074,  2562,  3652,  1012,
         1005,  2065,  2017,  2253,  1998,  7039,  1996,  6597,  1998, 16879,
         2408,  1996,  2406,  2017,  2071,  2763,  2424,  1037,  2751,  7529,
         2055,  3960,  1005,  1055,  2946,  1010,  2021,  2383,  2028,  1999,
         1037,  4951,  2066,  2023,  2003,  2087,  5866,  1998,  1045,  2031,
         2025,  2657,  1997], dtype=torch.int32)

# Notes:
- from above test, we can see that a token embedding in plaid, is saved as codes (to recover centroid+residual)
- codes =/= wordpiece tid
- for each psg, the len(codes)=len(wpids)

In [21]:
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import pyterrier as pt
import math
from collections import defaultdict
# import json

# with open("./idf_map.json", "r", encoding="utf-8") as f:
#     idf_map = json.load(f)
    
stats ={'N': 8841823, 'eps': 1.0, 'add_one': True}


        
def qe_before_candidate_generation(
    factory, 
    idf_map, N_global, eps=1.0, add_one=True,
    top_psg=3, top_exp=10, beta=0.5, idf_alpha=1.0,
    # --- NEW: stop-word filtering (optional) ---
    stop_wp_ids=None,           # set[int] of WordPiece IDs for stop words
    wp_sidecar=None             # object with .lookup_wpids(pids) -> (wpids_1d, lens)
):
    """
    不取原文：直接用索引解压得到 token 向量 V 和对应 codes，
    用 score = MaxSim(Q,V) * IDF(code)^idf_alpha 打分，选前 top_exp 扩展向量。
    若提供 stop_wp_ids 和 wp_sidecar，则在打分前进行基于 WordPiece ID 的停用词过滤。
    """
    embS = factory.searcher.ranker.embeddings_strided
    default_idf = math.log((N_global + eps) / (1 + eps)) + (1.0 if add_one else 0.0)

    def _expand(dfq):
        qid, qtext = dfq.iloc[0]["qid"], dfq.iloc[0]["query"]

        # 1) Obtain query embeddings
        Q = factory.searcher.encode([qtext]).squeeze(0)     # [Lq, dim]
        Q = F.normalize(Q, p=2, dim=-1)

        # 2) PRF：take first k as prf passages, obtain the PRF pids
        pids, _, _ = factory.searcher.dense_search(Q.unsqueeze(0), k=top_psg)

        # 3) Fetch all the PRF pids's corresponding vectors: V; and the their codes
        V, lens  = embS.lookup_pids(pids)       # [sumL, dim]
        if V is None or V.numel() == 0:
            return pd.DataFrame([{"qid": qid, "query": qtext, "query_vec": Q.unsqueeze(0)}])
        V = F.normalize(V.to(torch.float32), p=2, dim=-1)

        codes, _ = embS.lookup_codes(pids)      # [sumL]
        assert int(codes.numel()) == int(V.shape[0])

        # --- NEW: 基于 WP id 的停用词过滤（零文本 I/O） ---
        # filter the stopword's codes
        if (stop_wp_ids is not None) and (wp_sidecar is not None):
            wpids, _ = wp_sidecar.lookup_wpids(pids)       # [sumL]
            if wpids.numel() == V.size(0):
                mask = ~torch.tensor([int(x) in stop_wp_ids for x in wpids.tolist()],
                                     dtype=torch.bool, device=V.device)
                if mask.any():
                    V = V[mask]
                    codes = codes[mask]
                else:
                    # if all stopwords, then return only query
                    return pd.DataFrame([{"qid": qid, "query": qtext, "query_vec": Q.unsqueeze(0)}])

        # 4) score = MaxSim(Q,V) * (IDF(code))^idf_alpha
        sim = (Q @ V.T).max(dim=0).values
        idf_vals = [idf_map.get(int(c), default_idf) for c in codes.cpu().tolist()]
        idf_vec  = torch.tensor(idf_vals, dtype=V.dtype, device=V.device)
        score    = sim * (idf_vec ** float(idf_alpha))

        # 5) expand to the query
        k = min(top_exp, V.size(0))
        top_idx = torch.topk(score, k=k).indices
        E = beta * V[top_idx]                     # [E, dim]
        Q_new = torch.cat([Q, E], dim=0).unsqueeze(0)

        return pd.DataFrame([{"qid": qid, "query": qtext, "query_vec": Q_new}])

    return pt.apply.by_query(_expand, add_ranks=False)


In [23]:
def plaid_end_to_end_qe(factory, k=1000) -> pt.Transformer:
    assert factory.plaid_mode == True, "plaid_end_to_end should only be used in PLAID mode"
    def _search(df_query):
        assert len(df_query) == 1
        query_text = df_query.iloc[0]["query"]
        qid = df_query.iloc[0]["qid"]
        row = df_query.iloc[0]
        # Encode the query
        if 'query_vec' in df_query.columns:
            query_vec = row.query_vec
            if isinstance(query_vec, np.ndarray):
                Q = torch.from_numpy(query_vec)
            if torch.cuda.is_available():
                Q = query_vec.to('cuda', non_blocking=True).to(dtype=torch.float16)
        else:
            Q = factory.searcher.encode([query_text])
        assert Q.dim() == 3
        # Q = factory.searcher.encode([query_text])
        

        # PLAID candidate generation: returns pids and centroid_scores
        # https://github.com/stanford-futuredata/ColBERT/blob/main/colbert/search/candidate_generation.py#L45
        pids, centroid_scores = factory.searcher.ranker.generate_candidates(
            factory.searcher.config, Q
        )
        # PLAID centroid interaction, pruning and final scoring
        # score_pids returns (scores, pids)
        # https://github.com/stanford-futuredata/ColBERT/blob/main/colbert/search/index_storage.py#L111
        scores, pids = factory.searcher.ranker.score_pids(
            factory.searcher.config, Q, pids, centroid_scores
        )
               
        # Extract the top-k results
        topk = min(k, len(pids))
        top_indices = scores.argsort(descending=True)[:topk]
        results = []
        for rank, idx in enumerate(top_indices):
            pid = pids[idx].item()
            docno = factory.docnos.fwd[pid]
            # docno = self.docno_mapping.get(pid, "unknown_docno")
            score = scores[idx].item()
            results.append([qid, docno, score, rank + 1])

        return pd.DataFrame(results, columns=["qid", "docno", "score", "rank"])

    return pt.apply.by_query(_search)

# with stopwords removal: the best res below
## 1. using QE improve ndcg@10

## 2. perform centroid  recall boost to improve recall: use only the centroid frequency as centroid importance score


In [24]:
def _ivf_lookup_pids(ranker):
    """
    requirements: index contains: ivf.pid.pt file
    Returns  lookup_pids(c_ids) -> (packed_pids: LongTensor, lengths: LongTensor)
    """
    ivf_pid = getattr(ranker, 'ivf_pid', None)
    tbl = ivf_pid if ivf_pid is not None else ranker.ivf  # 两者都返回 PID

    def lookup_pids(c_ids):
        return tbl.lookup(c_ids)

    return lookup_pids
def centroid_recall_boost(factory, prf_k=20, m_extra=40, k=3000):
    """
    prf_k,          # mine centroid usage from top-20 base results
    m_extra,        # add 4 extra centroids (not in original ncells)
    rounds,         # iteration 
    k=1000          # final cutoff
    """
    import pandas as pd, torch, torch.nn.functional as F
    def l2(x): return F.normalize(x, p=2, dim=-1)

    ranker = factory.searcher.ranker
    cfg    = factory.searcher.config
    C      = ranker.codec.centroids                     # [C, dim]
    embS   = ranker.embeddings_strided
    lookup_pids = _ivf_lookup_pids(ranker)

    def _run(dfq):
        qid = dfq.iloc[0]["qid"]

        # 1)check is query_vec exists
        if "query_vec" in dfq.columns and dfq.iloc[0]["query_vec"] is not None:
            Q = dfq.iloc[0]["query_vec"].squeeze(0).to(torch.float32)  # [L, dim]
        else:
            qtext = dfq.iloc[0]["query"]
            Q = factory.searcher.encode([qtext]).squeeze(0).to(torch.float32)
        Q = l2(Q); device = Q.device

        # 2) baseline pids
        base_pids, _, base_scores = factory.searcher.dense_search(Q.unsqueeze(0), k=k)
        base_pids_list = base_pids
        base_scores_t  = torch.tensor(base_scores, device=device, dtype=torch.float32)

        # 3) used cells
        C_dev = C.to(device=device, dtype=torch.float32)
        sims  = (C_dev @ Q.T).max(dim=1).values
        used  = set(torch.topk(sims, k=min(cfg.ncells, C_dev.size(0))).indices.tolist())

        # 4) take the prf_k  PRF the high frequency cell
        prf = base_pids_list[:min(prf_k, len(base_pids_list))]
        freq = {}
        if prf:
            D_packed, D_lens = embS.lookup_pids(prf)  # CPU
            off = 0
            for L in D_lens.tolist():
                if L == 0: continue
                D = D_packed[off:off+L].to(device=device, dtype=torch.float32)
                assign = (C_dev @ D.T).argmax(dim=0)   # most closest centroids
                for c in assign.tolist():
                    freq[c] = freq.get(c, 0) + 1
                off += L

        extra = [c for c,_ in sorted(freq.items(), key=lambda x: x[1], reverse=True)
                 if c not in used][:m_extra]

        # 5) 拉取这些 extra cells 的倒排，评分并合并
        pid_chunks = []
        for c in extra:
            p_packed, p_lens = lookup_pids([int(c)])  # CPU posting list
            n = int(p_lens[0])
            if n == 0: continue
            pid_chunks.append(p_packed[:n])

        if pid_chunks:
            new_pids_all = torch.cat(pid_chunks).long()
            base_set = set(base_pids_list)
            keep = [pid for pid in new_pids_all.tolist() if pid not in base_set]
            if keep:
                new_pids = torch.tensor(keep, dtype=torch.long, device=device)
                scores2, pids2 = ranker.score_pids(cfg, Q.unsqueeze(0), new_pids, centroid_scores=None)
                all_pids   = torch.cat([torch.tensor(base_pids_list, device=device), pids2])
                all_scores = torch.cat([base_scores_t, scores2])
                best = {}
                for pid, s in zip(all_pids.tolist(), all_scores.tolist()):
                    if pid not in best or s > best[pid]:
                        best[pid] = s
                merged = sorted(best.items(), key=lambda x: x[1], reverse=True)[:k]
                base_pids_list = [pid for pid,_ in merged]
                base_scores_t  = torch.tensor([s for _,s in merged], device=device)

        docnos = factory.docnos.fwd[base_pids_list]
        return pd.DataFrame({
            "qid":   [qid]*len(docnos),
            "docno": docnos,
            "score": base_scores_t[:len(docnos)].cpu().numpy(),
            "rank":  list(range(1, len(docnos)+1))
        })

    import pyterrier as pt
    return pt.apply.by_query(_run, add_ranks=False)


In [25]:
import pandas as pd
import pyterrier as pt

class RRFMerger(pt.Transformer):
    def __init__(self, pipes, k=60, score_col="score", rank_col="rank"):
        super().__init__()
        self.pipes = pipes
        self.k = k
        self.score_col = score_col
        self.rank_col = rank_col

    def _to_rrf(self, df):
        out = df[["qid", "docno"]].copy()
        if self.rank_col in df.columns:
            out["rrf"] = 1.0 / (self.k + df[self.rank_col].astype(float))
        else:
            # 若没有 rank，就按 score 降序生成 per-qid 排名
            tmp = df[["qid", self.score_col]].copy()
            tmp["rr"] = tmp.groupby("qid")[self.score_col].rank(ascending=False, method="first")
            out["rrf"] = 1.0 / (self.k + tmp["rr"])
        return out

    def transform(self, topics):
        # 逐路跑检索器
        dfs = [p.transform(topics) for p in self.pipes]
        # 计算每一路的 RRF 分数并相加
        rrfs = [self._to_rrf(df) for df in dfs]
        fused = pd.concat(rrfs).groupby(["qid","docno"], as_index=False)["rrf"].sum()
        fused = fused.sort_values(["qid","rrf"], ascending=[True, False])
        fused["rank"] = fused.groupby("qid").cumcount() + 1
        fused = fused.rename(columns={"rrf":"score"})
        return fused


In [27]:
boost = centroid_recall_boost(
    factory,
    prf_k=3,          # mine centroid usage from top-20 base results
    m_extra=50,         # add 4 extra centroids (not in original ncells)
    k=1000             # final cutoff
)

plaid_e2e = factory.plaid_end_to_end(k=1000)
from  pyterrier.measures import *
res_dl20 = pt.Experiment(
    [plaid_e2e, boost],
    topics2019, qrels2019,
    eval_metrics = [RR(rel=2), nDCG@10, nDCG@100, AP(rel=2),R(rel=2)@10, R(rel=2)@100, R(rel=2)@1000],
    names=["plaid", "plaid+centroid boost"], verbose = True
)
res_dl20

pt.Experiment: 100%|██████████████████████████| 2/2 [00:25<00:00, 12.64s/system]


,name,RR(rel=2),nDCG@10,nDCG@100,AP(rel=2),R(rel=2)@10,R(rel=2)@100,R(rel=2)@1000
0,plaid,0.893411,0.738266,0.683256,0.509709,0.288459,0.665104,0.870589
1,plaid+centroid boost,0.893411,0.738266,0.684389,0.511493,0.288459,0.666415,0.896836


In [118]:
from pyterrier.measures import *
res1 = pt.Experiment(
    [baseline, qe_stage>> plaid_end_to_end_qe(factory, k=1000) , boost_stage, pipeline_qe_boost, pipe_rrf],
    topics2020, qrels2020,
    eval_metrics=[RR(rel=2), nDCG@10, nDCG@100, nDCG@1000, AP(rel=2),
                  R(rel=2)@10, R(rel=2)@100, R(rel=2)@1000, "num_ret"],
    names=['plaid','QE','boost', 'QE+centroidBoost','RRF(plaid,QE+boost)'],
    verbose=True
)
res1

pt.Experiment: 100%|██████████████████████████| 5/5 [01:38<00:00, 19.70s/system]


,name,RR(rel=2),nDCG@10,nDCG@100,nDCG@1000,AP(rel=2),R(rel=2)@10,R(rel=2)@100,R(rel=2)@1000,num_ret
0,plaid,0.842306,0.740983,0.696068,0.766168,0.526206,0.402467,0.770992,0.904436,54000.0
1,QE,0.846297,0.758560,0.702624,0.767557,0.532543,0.403037,0.774504,0.899703,54000.0
2,boost,0.842306,0.740983,0.697948,0.771593,0.527647,0.402467,0.772661,0.910765,54000.0
3,QE+centroidBoost,0.845370,0.759567,0.697749,0.768103,0.530307,0.403621,0.766257,0.903846,54000.0
4,"RRF(plaid,QE+boost)",0.838722,0.754511,0.701806,0.775596,0.531864,0.403915,0.773190,0.913149,73973.0


In [121]:
from pyterrier.measures import *
res2 = pt.Experiment(
    [baseline, qe_stage>> plaid_end_to_end_qe(factory, k=1000) , boost_stage, pipeline_qe_boost, pipe_rrf],
    topics_dlhard, qrels_dlhard,
    eval_metrics=[RR(rel=2), nDCG@10, nDCG@100, nDCG@1000, AP(rel=2),
                  R(rel=2)@10, R(rel=2)@100, R(rel=2)@1000, "num_ret"],
    names=['plaid','QE','boost', 'QE+centroidBoost','RRF(plaid,QE+boost)'],
    verbose=True
)
res2

pt.Experiment: 100%|██████████████████████████| 5/5 [01:41<00:00, 20.23s/system]


,name,RR(rel=2),nDCG@10,nDCG@100,nDCG@1000,AP(rel=2),R(rel=2)@10,R(rel=2)@100,R(rel=2)@1000,num_ret
0,plaid,0.581308,0.414594,0.460658,0.535359,0.272702,0.289868,0.641487,0.838944,50000.0
1,QE,0.579651,0.426992,0.465016,0.535373,0.278470,0.281472,0.639194,0.848302,50000.0
2,boost,0.581301,0.414594,0.460903,0.542127,0.273276,0.289868,0.642132,0.867931,50000.0
3,QE+centroidBoost,0.559626,0.422686,0.457459,0.529084,0.272540,0.275742,0.634121,0.853338,50000.0
4,"RRF(plaid,QE+boost)",0.561482,0.422107,0.466677,0.539983,0.276321,0.279330,0.650115,0.856298,67592.0


# some other results: parameter tuning etcs

In [151]:
# 在线QE阶段
qe_stage = qe_before_candidate_generation(
    factory,
    idf_map=idf_map, N_global=stats["N"], eps=1.0, add_one=True,
    top_psg=6, top_exp=60, beta=0.5, idf_alpha=1,)
    stop_wp_ids=STOP_WP_IDS, wp_sidecar=wp_sidecar   # ← 仅此两参即可启用过滤
)
# pipe = qe_before_candidate_generation(factory,top_psg=3,top_exp=20,beta=0.2) >>plaid_end_to_end_qe(factory, k=1000)
# Use PyTerrier to evaluate and compare the effectiveness metrics

plaid_e2e = factory.plaid_end_to_end(k=1000)
from pyterrier.measures import *
import torch
experiment_results5 = pt.Experiment(
    [plaid_e2e,qe_stage >> plaid_end_to_end_qe(factory, k=1000) ],
    topics2019,
    qrels2019,
    eval_metrics=[RR(rel=2), nDCG@10, nDCG@100,nDCG@1000, AP(rel=2),R(rel=2)@10, R(rel=2)@100, R(rel=2)@1000,"num_ret"],
    names=[ 'plaid_e2e','plaid_C'], verbose=True,
)

experiment_results5

pt.Experiment: 100%|██████████████████████████| 2/2 [00:13<00:00,  6.51s/system]


,name,RR(rel=2),nDCG@10,nDCG@100,nDCG@1000,AP(rel=2),R(rel=2)@10,R(rel=2)@100,R(rel=2)@1000,num_ret
0,plaid_e2e,0.893411,0.738266,0.683256,0.756087,0.509709,0.288459,0.665104,0.870589,43000.0
1,plaid_C,0.907623,0.752289,0.694422,0.767791,0.529610,0.291173,0.673487,0.881633,43000.0


In [88]:
# 在线QE阶段
qe_stage = qe_before_candidate_generation_codesidf_sw(
    factory,
    idf_map=idf_map, N_global=stats["N"], eps=1.0, add_one=True,
    top_psg=6, top_exp=65, beta=0.5, idf_alpha=1,
    stop_wp_ids=STOP_WP_IDS, wp_sidecar=wp_sidecar   # ← 仅此两参即可启用过滤
)
# pipe = qe_before_candidate_generation(factory,top_psg=3,top_exp=20,beta=0.2) >>plaid_end_to_end_qe(factory, k=1000)
# Use PyTerrier to evaluate and compare the effectiveness metrics

plaid_e2e = factory.plaid_end_to_end(k=1000)
from pyterrier.measures import *
import torch
experiment_results5 = pt.Experiment(
    [plaid_e2e,qe_stage >> plaid_end_to_end_qe(factory, k=1000) ],
    topics2019,
    qrels2019,
    eval_metrics=[RR(rel=2), nDCG@10, nDCG@100,nDCG@1000, AP(rel=2),R(rel=2)@10, R(rel=2)@100, R(rel=2)@1000,"num_ret"],
    names=[ 'plaid_e2e','plaid_C'], verbose=True,
)

experiment_results5

pt.Experiment: 100%|██████████████████████████| 2/2 [00:12<00:00,  6.44s/system]


,name,RR(rel=2),nDCG@10,nDCG@100,nDCG@1000,AP(rel=2),R(rel=2)@10,R(rel=2)@100,R(rel=2)@1000,num_ret
0,plaid_e2e,0.893411,0.738266,0.683256,0.756087,0.509709,0.288459,0.665104,0.870589,43000.0
1,plaid_C,0.905869,0.752008,0.693596,0.766433,0.524551,0.295855,0.683326,0.885338,43000.0


In [83]:
plaid_e2e = factory.plaid_end_to_end(k=1000)
systems = [plaid_e2e]
names = ['plaid']
for  m in range(1,10):
    qe_stage = qe_before_candidate_generation_codesidf_sw(    factory,
    idf_map=idf_map, N_global=stats["N"], eps=1.0, add_one=True,
    top_psg=m, top_exp=50, beta=0.4, idf_alpha=1,
    stop_wp_ids=STOP_WP_IDS, wp_sidecar=wp_sidecar   # ← 仅此两参即可启用过滤
)
    plaid_e2e_qe = plaid_end_to_end_qe(factory, k=1000)
    systems.append(qe_stage>>plaid_e2e_qe)
    
    names.append(f'C_qe_prf_k={m}-SW')
from pyterrier.measures import *
res_m_psg = pt.Experiment(
    systems,
    topics2019, qrels2019,
    eval_metrics = [RR(rel=2), nDCG@10, nDCG@100, nDCG@1000,AP(rel=2),R(rel=2)@10, R(rel=2)@100, R(rel=2)@1000,"num_ret"],
    names=names, verbose = True
)
res_m_psg


pt.Experiment: 100%|████████████████████████| 10/10 [01:13<00:00,  7.36s/system]


,name,RR(rel=2),nDCG@10,nDCG@100,nDCG@1000,AP(rel=2),R(rel=2)@10,R(rel=2)@100,R(rel=2)@1000,num_ret
0,plaid,0.893411,0.738266,0.683256,0.756087,0.509709,0.288459,0.665104,0.870589,43000.0
1,C_qe_prf_k=1-SW,0.884628,0.749862,0.693878,0.765751,0.535674,0.294303,0.675903,0.881777,43000.0
2,C_qe_prf_k=2-SW,0.880244,0.745000,0.688476,0.757576,0.520703,0.285846,0.680780,0.872181,43000.0
3,C_qe_prf_k=3-SW,0.898117,0.742094,0.691257,0.764629,0.518514,0.279268,0.685215,0.883136,43000.0
4,C_qe_prf_k=4-SW,0.887043,0.748501,0.694399,0.765280,0.523627,0.293804,0.683725,0.876467,43000.0
5,C_qe_prf_k=5-SW,0.896179,0.748789,0.698297,0.768799,0.529616,0.292579,0.683600,0.875889,43000.0
6,C_qe_prf_k=6-SW,0.883167,0.752768,0.696570,0.769357,0.526696,0.296976,0.678048,0.875696,43000.0
7,C_qe_prf_k=7-SW,0.883167,0.748317,0.698901,0.770044,0.527008,0.293024,0.682543,0.877457,43000.0
8,C_qe_prf_k=8-SW,0.881395,0.746496,0.697020,0.767328,0.526096,0.292831,0.675437,0.872467,43000.0
9,C_qe_prf_k=9-SW,0.859432,0.737332,0.693731,0.761913,0.521235,0.289691,0.674398,0.867656,43000.0


In [84]:
plaid_e2e = factory.plaid_end_to_end(k=1000)
systems = [plaid_e2e]
names = ['plaid']
for  m in range(1,101,10):
    qe_stage = qe_before_candidate_generation_codesidf_sw(    factory,
    idf_map=idf_map, N_global=stats["N"], eps=1.0, add_one=True,
    top_psg=6, top_exp=m, beta=0.4, idf_alpha=1,
    stop_wp_ids=STOP_WP_IDS, wp_sidecar=wp_sidecar   # ← 仅此两参即可启用过滤
)
    plaid_e2e_qe = plaid_end_to_end_qe(factory, k=1000)
    systems.append(qe_stage>>plaid_e2e_qe)
    
    names.append(f'C_qe_prf_k={m}-SW')
from pyterrier.measures import *
res_m_psg = pt.Experiment(
    systems,
    topics2019, qrels2019,
    eval_metrics = [RR(rel=2), nDCG@10, nDCG@100, nDCG@1000,AP(rel=2),R(rel=2)@10, R(rel=2)@100, R(rel=2)@1000,"num_ret"],
    names=names, verbose = True
)
res_m_psg


pt.Experiment: 100%|████████████████████████| 11/11 [01:21<00:00,  7.40s/system]


,name,RR(rel=2),nDCG@10,nDCG@100,nDCG@1000,AP(rel=2),R(rel=2)@10,R(rel=2)@100,R(rel=2)@1000,num_ret
0,plaid,0.893411,0.738266,0.683256,0.756087,0.509709,0.288459,0.665104,0.870589,43000.0
1,C_qe_prf_k=1-SW,0.877907,0.734891,0.682865,0.755610,0.508250,0.285136,0.667338,0.870195,43000.0
2,C_qe_prf_k=11-SW,0.868605,0.737427,0.690857,0.760100,0.515156,0.289683,0.674416,0.870978,43000.0
3,C_qe_prf_k=21-SW,0.897508,0.744577,0.698946,0.767215,0.520967,0.291326,0.683258,0.875130,43000.0
4,C_qe_prf_k=31-SW,0.872868,0.744092,0.698535,0.767422,0.524012,0.291687,0.682253,0.873639,43000.0
5,C_qe_prf_k=41-SW,0.868217,0.746073,0.697971,0.769023,0.525653,0.292792,0.679352,0.876023,43000.0
6,C_qe_prf_k=51-SW,0.882752,0.753284,0.697035,0.769094,0.526964,0.297104,0.678620,0.875665,43000.0
7,C_qe_prf_k=61-SW,0.908361,0.754173,0.696656,0.768046,0.529765,0.294495,0.680672,0.880195,43000.0
8,C_qe_prf_k=71-SW,0.906423,0.750451,0.693366,0.766609,0.525196,0.294631,0.682872,0.885671,43000.0
9,C_qe_prf_k=81-SW,0.893033,0.745572,0.691317,0.764145,0.523239,0.290008,0.679236,0.886557,43000.0


In [85]:
plaid_e2e = factory.plaid_end_to_end(k=1000)
systems = [plaid_e2e]
names = ['plaid']
for  m in range(1,11,1):
    b= m/10
    qe_stage = qe_before_candidate_generation_codesidf_sw(    factory,
    idf_map=idf_map, N_global=stats["N"], eps=1.0, add_one=True,
    top_psg=6, top_exp=60, beta=b, idf_alpha=1,
    stop_wp_ids=STOP_WP_IDS, wp_sidecar=wp_sidecar   # ← 仅此两参即可启用过滤
)
    plaid_e2e_qe = plaid_end_to_end_qe(factory, k=1000)
    systems.append(qe_stage>>plaid_e2e_qe)
    
    names.append(f'C_qe_prf_k={m}-SW')
from pyterrier.measures import *
res_m_psg = pt.Experiment(
    systems,
    topics2019, qrels2019,
    eval_metrics = [RR(rel=2), nDCG@10, nDCG@100, nDCG@1000,AP(rel=2),R(rel=2)@10, R(rel=2)@100, R(rel=2)@1000,"num_ret"],
    names=names, verbose = True
)
res_m_psg


pt.Experiment: 100%|████████████████████████| 11/11 [01:21<00:00,  7.43s/system]


,name,RR(rel=2),nDCG@10,nDCG@100,nDCG@1000,AP(rel=2),R(rel=2)@10,R(rel=2)@100,R(rel=2)@1000,num_ret
0,plaid,0.893411,0.738266,0.683256,0.756087,0.509709,0.288459,0.665104,0.870589,43000.0
1,C_qe_prf_k=1-SW,0.881229,0.741389,0.693802,0.765291,0.517851,0.285556,0.675902,0.877269,43000.0
2,C_qe_prf_k=2-SW,0.897287,0.746612,0.695581,0.768348,0.523614,0.288805,0.678018,0.878832,43000.0
3,C_qe_prf_k=3-SW,0.896733,0.751828,0.696788,0.768410,0.526958,0.294868,0.679749,0.877811,43000.0
4,C_qe_prf_k=4-SW,0.907946,0.753079,0.696510,0.767665,0.529629,0.293484,0.680695,0.874512,43000.0
5,C_qe_prf_k=5-SW,0.907623,0.752289,0.694422,0.767791,0.529610,0.291173,0.673487,0.881633,43000.0
6,C_qe_prf_k=6-SW,0.907623,0.750740,0.693392,0.768705,0.527147,0.290435,0.672435,0.885726,43000.0
7,C_qe_prf_k=7-SW,0.907069,0.749831,0.692670,0.767986,0.526134,0.289327,0.672209,0.886404,43000.0
8,C_qe_prf_k=8-SW,0.905906,0.750499,0.692791,0.766666,0.525325,0.290551,0.673251,0.885506,43000.0
9,C_qe_prf_k=9-SW,0.905906,0.750449,0.692135,0.765611,0.524638,0.290551,0.673693,0.884470,43000.0


In [10]:
# 在线QE阶段
qe_stage = qe_before_candidate_generation_codesidf(
    factory,
    idf_map=idf_map,
    N_global=stats["N"], eps=stats["eps"], add_one=stats["add_one"],
    top_psg=5, top_exp=115, beta=0.5, idf_alpha=1.0
)
# pipe = qe_before_candidate_generation(factory,top_psg=3,top_exp=20,beta=0.2) >>plaid_end_to_end_qe(factory, k=1000)
# Use PyTerrier to evaluate and compare the effectiveness metrics

plaid_e2e = factory.plaid_end_to_end(k=1000)
from pyterrier.measures import *
import torch
experiment_results5 = pt.Experiment(
    [plaid_e2e,qe_stage >> plaid_end_to_end_qe(factory, k=1000) ],
    topics2019,
    qrels2019,
    eval_metrics=[RR(rel=2), nDCG@10, nDCG@100,nDCG@1000, AP(rel=2),R(rel=2)@10, R(rel=2)@100, R(rel=2)@1000,"num_ret"],
    names=[ 'plaid_e2e','plaid_C'], verbose=True,baseline=0
)

experiment_results5

pt.Experiment:   0%|                                  | 0/2 [00:00<?, ?system/s]


#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: who is robert gray, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([ 101,    1, 2040, 2003, 2728, 3897,  102,  103,  103,  103,  103,  103,
         103,  103,  103,  103,  103,  103,  103,  103,  103,  103,  103,  103,
         103,  103,  103,  103,  103,  103,  103,  103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')



pt.Experiment: 100%|██████████████████████████| 2/2 [00:15<00:00,  7.59s/system]


,name,RR(rel=2),nDCG@10,nDCG@100,nDCG@1000,AP(rel=2),R(rel=2)@10,R(rel=2)@100,R(rel=2)@1000,num_ret
0,plaid_e2e,0.893411,0.738266,0.683256,0.756087,0.509709,0.288459,0.665104,0.870589,43000.0
1,plaid_C,0.912957,0.749018,0.685378,0.746037,0.525479,0.289628,0.674319,0.852572,43000.0


In [90]:
# 在线QE阶段
qe_stage = qe_before_candidate_generation_codesidf(
    factory,
    idf_map=idf_map,
    N_global=stats["N"], eps=stats["eps"], add_one=stats["add_one"],
    top_psg=6, top_exp=70, beta=0.5, idf_alpha=1.0
)
# pipe = qe_before_candidate_generation(factory,top_psg=3,top_exp=20,beta=0.2) >>plaid_end_to_end_qe(factory, k=1000)
# Use PyTerrier to evaluate and compare the effectiveness metrics

plaid_e2e = factory.plaid_end_to_end(k=1000)
from pyterrier.measures import *
import torch
experiment_results5 = pt.Experiment(
    [plaid_e2e,qe_stage >> plaid_end_to_end_qe(factory, k=1000) ],
    topics2019,
    qrels2019,
    eval_metrics=[RR(rel=2), nDCG@10, nDCG@100,nDCG@1000, AP(rel=2),R(rel=2)@10, R(rel=2)@100, R(rel=2)@1000,"num_ret"],
    names=[ 'plaid_e2e','plaid_C'], verbose=True
)

experiment_results5

pt.Experiment: 100%|██████████████████████████| 2/2 [00:12<00:00,  6.47s/system]


,name,RR(rel=2),nDCG@10,nDCG@100,nDCG@1000,AP(rel=2),R(rel=2)@10,R(rel=2)@100,R(rel=2)@1000,num_ret
0,plaid_e2e,0.893411,0.738266,0.683256,0.756087,0.509709,0.288459,0.665104,0.870589,43000.0
1,plaid_C,0.904393,0.750636,0.691488,0.766873,0.523057,0.290623,0.678381,0.886831,43000.0


In [28]:
# 在线QE阶段
qe_stage = qe_before_candidate_generation_codesidf(
    factory,
    idf_map=idf_map,
    N_global=stats["N"], eps=stats["eps"], add_one=stats["add_one"],
    top_psg=6, top_exp=70, beta=0.5, idf_alpha=1.0
)
# pipe = qe_before_candidate_generation(factory,top_psg=3,top_exp=20,beta=0.2) >>plaid_end_to_end_qe(factory, k=1000)
# Use PyTerrier to evaluate and compare the effectiveness metrics

plaid_e2e = factory.plaid_end_to_end(k=1000)
from pyterrier.measures import *
import torch
experiment_results5 = pt.Experiment(
    [plaid_e2e,qe_stage >> plaid_end_to_end_qe(factory, k=1000) ],
    topics2019,
    qrels2019,
    eval_metrics=[RR(rel=2), nDCG@10, nDCG@100,nDCG@1000, AP(rel=2),R(rel=2)@10, R(rel=2)@100, R(rel=2)@1000,"num_ret"],
    names=[ 'plaid_e2e','plaid_newtry'], verbose=True
)

experiment_results5

pt.Experiment: 100%|██████████████████████████| 2/2 [00:13<00:00,  6.52s/system]


,name,RR(rel=2),nDCG@10,nDCG@100,nDCG@1000,AP(rel=2),R(rel=2)@10,R(rel=2)@100,R(rel=2)@1000,num_ret
0,plaid_e2e,0.893411,0.738266,0.683256,0.756087,0.509709,0.288459,0.665104,0.870589,43000.0
1,plaid_newtry,0.904393,0.750636,0.691488,0.766873,0.523057,0.290623,0.678381,0.886831,43000.0
